In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'        # Reduce TensorFlow log messages
os.environ["TF_DETERMINISTIC_OPS"] = "1"        # Enable deterministic operations for reproducibility
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"      # Fix cudnn randomness for reproducibility
# 依赖导入
import random
import numpy as np
import tensorflow as tf
from pathlib import Path
from model import init_model, fun

In [ ]:
import cfg
cell = "GM12878"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
SEED = 42  # 5 sets of random seeds:42,43,44,45,46
root_dir = "/mnt/d/TAD-MultiOmicsNet"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

use_epi = cfg.model_cfg["use_epi"]
use_dnabert = cfg.model_cfg["use_dnabert"]
use_curv = cfg.model_cfg["use_curv"]
use_transformer = cfg.model_cfg["use_transformer"]
use_maskconv = cfg.model_cfg["use_maskconv"]
use_gaussian = cfg.model_cfg["use_gaussian"]
use_adaptive_fuse = cfg.model_cfg["use_adaptive_fuse"]
model_use = cfg.get_model_name(SEED)

In [ ]:
### 1. Prediction Module: Load trained model, run inference on test chromosomes, save predicted probability to npy files.

In [ ]:

out_dir = Path(f'{root_dir}/{cell}/{display_reso}kb/prediction_result/{model_use}')
out_dir.mkdir(parents=True, exist_ok=True)

model = init_model(
        use_epi=use_epi,
        use_dnabert=use_dnabert,
        use_curv=use_curv,
        use_transformer=use_transformer,
        use_maskconv=use_maskconv,
        use_gaussian=use_gaussian,
        use_adaptive_fuse=use_adaptive_fuse
        )
weight_path = f'{root_dir}/{cell}/{display_reso}kb/weights/new/{model_use}_best.h5'
model.load_weights(weight_path)
print(f"Weights loaded successfully：{weight_path}")

chrs = ["20", "21", "22"]  # GM12878 test set

epi_feat_all, epi_mask_all = None, None
dna_feat_all, dna_mask_all = None, None
curv_feat_all, curv_mask_all = None, None

ptr = 0
if use_epi:
    epi_feat_all = np.load(
        f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/epi{cell}_predict_feat100.npy").astype(
        "float32")
    epi_mask_all = np.load(
        f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/epi{cell}_predict_mask100.npy").astype(
        "float32")
    print(f"Global epigenetic feature shape: {epi_feat_all.shape}, mask shape:{epi_mask_all.shape}")

if use_dnabert:
    dna_feat_all = np.load(
        f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/dna{cell}_predict_{display_reso}kb.npy").astype(
        "float32")
    dna_mask_all = np.load(
        f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/dna{cell}_predict_mask_{display_reso}kb.npy").astype(
        "float32")
    print(f"Global DNABERT sequence feature shape: {dna_feat_all.shape}, mask shape:{dna_mask_all.shape}")

if use_curv:
    curv_feat_all = np.load(
        f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/curv{cell}_predict_feat100.npy").astype(
        "float32")
    curv_mask_all = np.load(
        f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/curv{cell}_predict_mask100.npy").astype(
        "float32")
    print(f"Global DNA curvature feature shape: {curv_feat_all.shape}, mask shape:{curv_mask_all.shape}")

for chrom in chrs:
    npy_path = f'{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/{cell}_{display_reso}kb_chr{chrom}-matrix.npy'
    pre_data = np.load(npy_path)
    sample_num = pre_data.shape[0]
    print(f"\n[{cell}] {display_reso}kb chr{chrom} raw shape: {pre_data.shape} 样本数={sample_num}")

    data_hic_raw = pre_data[:, :-1].reshape(-1, 10, 10, 1).astype("float32")
    data_hic = fun(data_hic_raw)

    inputs = [data_hic]
    if use_epi:
        data_epi_feat = epi_feat_all[ptr:ptr + sample_num]
        data_epi_mask = epi_mask_all[ptr:ptr + sample_num]
        inputs.extend([data_epi_feat, data_epi_mask])
    if use_dnabert:
        data_dna_feat = dna_feat_all[ptr:ptr + sample_num]
        data_dna_mask = dna_mask_all[ptr:ptr + sample_num]
        inputs.extend([data_dna_feat, data_dna_mask])
    if use_curv:
        data_curv_feat = curv_feat_all[ptr:ptr + sample_num]
        data_curv_mask = curv_mask_all[ptr:ptr + sample_num]
        inputs.extend([data_curv_feat, data_curv_mask])
    ptr += sample_num

    # predict
    batch_size = 64
    predictions = model.predict(inputs, batch_size=batch_size, verbose=1)
    predictions1 = np.array(predictions.flatten())

    assert len(predictions1) == sample_num, f"The number of predicted samples for chr{chrom} does not match!"

    # Prediction Probability of Original Matrix Concatenation
    merged_data = np.concatenate((pre_data, predictions1.reshape(-1, 1)), axis=1)
    save_path = str(out_dir / f'{display_reso}kb_{cell}_chr{chrom}-pred.npy')
    np.save(save_path, merged_data)
    print(f"✅ 已保存 → {save_path}, shape={merged_data.shape}")

print("\nAll chromosome prediction tasks finished!")

In [ ]:
### 2. Evaluation Module: Load saved prediction files, compute classification metrics (Precision, Recall, F1, AUPRC, MCC)

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, matthews_corrcoef,
)


FIX_THRESH = 0.5
file_list = [
    ("chr20", f"{root_dir}/{cell}/{display_reso}kb/prediction_result/{model_use}/{display_reso}kb_{cell}_chr20-pred.npy"),
    ("chr21", f"{root_dir}/{cell}/{display_reso}kb/prediction_result/{model_use}/{display_reso}kb_{cell}_chr21-pred.npy"),
    ("chr22", f"{root_dir}/{cell}/{display_reso}kb/prediction_result/{model_use}/{display_reso}kb_{cell}_chr22-pred.npy"),
]

chr_prf1 = []
chr_auprc = []
chr_mcc = []
all_total_bins = 0

print("==== Per-chromosome evaluation ====")
for chr_name, path in file_list:
    arr = np.load(path)
    y_true = arr[:, -2]
    y_prob = arr[:, -1]
    y_pred = (y_prob >= FIX_THRESH)

    chr_total_bins = arr.shape[0]
    all_total_bins += chr_total_bins

    p = precision_score(y_true, y_pred, pos_label=1)
    r = recall_score(y_true, y_pred, pos_label=1)
    f1 = f1_score(y_true, y_pred, pos_label=1)
    mcc = matthews_corrcoef(y_true, y_pred)
    pos_num = np.sum(y_true == 1)
    pred_pos_num = np.sum(y_pred == 1)

    # AUPRC: skip if only one class exists
    if len(np.unique(y_true)) == 2:
        auprc = average_precision_score(y_true, y_prob)
    else:
        auprc = np.nan

    chr_prf1.append([p, r, f1])
    chr_auprc.append(auprc)
    chr_mcc.append(mcc)

    print(f"[{chr_name}] True boundaries:{pos_num};Pred boundaries:{pred_pos_num};P={p:.4f};R={r:.4f};F1={f1:.4f}")
    print(f"AUPRC={auprc:.4f};MCC={mcc:.4f}\n")

# Chromosome average metrics
chr_prf1_arr = np.array(chr_prf1)
chr_avg_p = np.nanmean(chr_prf1_arr[:, 0])
chr_avg_r = np.nanmean(chr_prf1_arr[:, 1])
chr_avg_f1 = np.nanmean(chr_prf1_arr[:, 2])
chr_avg_auprc = np.nanmean(chr_auprc)
chr_avg_mcc = np.nanmean(chr_mcc)

print("==== Chromosome average metrics (equal weight per chromosome) ====")
print(f"Average P={chr_avg_p:.4f};Average R={chr_avg_r:.4f};Average F1={chr_avg_f1:.4f}")
print(f"Average AUPRC={chr_avg_auprc:.4f};Average MCC={chr_avg_mcc:.4f}\n")